[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_3/L2.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 3 | BÀI 2


---

# **THỐNG KÊ BAYES TRONG THỰC TẾ**

|  |  |
|:---|:---|
|**Thời gian đọc** |  15 phút |
|**Kiến thức nền** |Phương pháp bộ phân loại Naïve Bayes, hàm hợp lý, phân phối tiên nghiệm.  |
|**Từ khóa** |Phân phối Gauss, Khớp mô hình, Hàm hợp lý, Phân phối tiên nghiệm  |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/"
FILES = ['mlfac_dat.csv']
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
        print("Đã tải:", f)

*Ở bài trước, ta đã xem thống kê Bayes có thể dùng để gán xác suất cho khả năng xảy ra của một kết quả như thế nào. Ta đã xét các đặc trưng phân loại, và bây giờ ta xét các đặc trưng liên tục để dự báo việc lợi suất một tháng tới của một nhóm cổ phiếu có vượt trung vị hay không.*

## **1. Naïve Bayes trong thực tế**

Hãy nhớ lại từ Module 3, Bài 1 rằng phân phối hậu nghiệm có thể biểu diễn bằng tích của các hàm hợp lý của các biến dự báo cho trước nhóm và các phân phối tiên nghiệm của nhóm, như trong (1.4). Điều này dựa trên giả định độc lập có điều kiện. Điều này cũng đúng với các biến dự báo là số liên tục, vì những biến dự báo này cũng có các phân phối hợp lý cho trước một nhóm mục tiêu cụ thể.

$$
\begin{align}
P(Y=k | X_{1}, X_{2}, \cdots , X_{n}) = \frac{P(X_{1}|Y=k)P(X_{2}|Y=k)\cdots P(X_{n}|Y=k)P(Y=k)}{\sum_{j}P(Y=j)\prod_{i=1}^{n}P(X_{i}|Y=j)}
\tag{2.1}
\end{align}
$$

Ta xét trường hợp phân phối hợp lý của biến dự báo tuân theo phân phối Gauss ở mỗi nhóm. Biến dự báo sẽ là biến giải thích mạnh nếu các phân phối khác nhau. Một biến dự báo mạnh, điều ta thường nhắm tới, có rất ít chồng lấn giữa các hàm hợp lý của từng nhóm, nhờ đó phân biệt được các nhóm. Điều này được thể hiện trong hình 1 cho cả biến dự báo mạnh và yếu. Biểu đồ trên cho thấy hàm hợp lý của một biến dự báo mạnh khi cho trước nhóm mục tiêu, vì có rất ít chồng lấn giữa các phân phối (hai đường cong). Biểu đồ dưới cho thấy một biến dự báo yếu hơn với mức chồng lấn lớn hơn đáng kể.

**Hình 1: Hàm hợp lý của biến dự báo mạnh và yếu**

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/M3Lesson2_plt1.png)

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/M3Lesson2_plt2.png)

Để hình dung rõ hơn các ý tưởng này, ta xét một ví dụ áp dụng bộ phân loại Naïve Bayes cho bài toán lợi suất cổ phiếu.

Trong phần này, ta dùng dữ liệu của Coqueret và Guida (chương 9, mục 9.4), cụ thể là dữ liệu lợi suất một tháng tới của một tập cổ phiếu và dữ liệu tài chính. Dữ liệu có thể truy cập tại https://github.com/shokru/mlfactor.github.io/tree/master/material. Để minh họa, ta lấy một tập con cổ phiếu và đặc trưng để dự báo việc giá một tháng tới có cao hơn trung vị hay không, tức là một bài toán phân loại. Các đặc trưng ta xem xét là:

* *Mkt_Cap_12M_Usd* - Quy mô
* *Pb* - Giá trên giá trị sổ sách
* *Vol1Y_Usd* - Mức quay vòng cổ phiếu
* *Return_On_Capital* - Lợi nhuận trên vốn
* *Roe* - Lợi nhuận trên vốn chủ sở hữu
* *Pe* - Tỷ lệ giá trên thu nhập

Ta thực hiện phân tích bằng phần mềm thống kê Python. Hãy nạp các thư viện cần thiết và nhập dữ liệu. Ta sẽ làm việc với dữ liệu từ năm 2000 đến 2018 và chỉ 50 mã cổ phiếu.

In [ ]:
import numpy as np
import pandas as pd

df_50 = pd.read_csv("mlfac_dat.csv")

# consider data from 2000 to 2018

df_50 = df_50[(df_50["date"] > "1999-12-31") & (df_50["date"] < "2019-01-01")]

# Filter on the first 50 stock ids

df_50 = df_50[df_50["stock_id"] <= 50]

Mục tiêu là dự báo xem lợi suất một tháng tới có cao hơn trung vị hay không. Do đó, ta chỉ cần gắn cờ 1 cho các trường hợp cao hơn trung vị và 0 cho các trường hợp còn lại.

In [ ]:
# Created a class variable for 1-Month returns if > median

df_50["R1M_Usd_C"] = np.where(df_50["R1M_Usd"] > df_50["R1M_Usd"].median(), 1, 0)

# Make the response variable into integer format

df_50["R1M_Usd_C"] = df_50["R1M_Usd_C"].astype(int)

Chỉ định các biến dự báo hay đặc trưng dùng trong bộ phân loại. Các đặc trưng này đã được biến đổi thành các biến tuân theo phân phối đều. Trong thực tế, ta mong muốn các biến phân phối Gauss. Vì vậy, ta sẽ biến đổi từng đặc trưng bằng phép biến đổi Box-Muller (Box và Muller 610).

In [ ]:
# create a copy of data to store Gaussian converted data

df50_norm = df_50.copy()

# Features of interest
feature_cols = ["Mkt_Cap_12M_Usd", "Pb", "Vol1Y_Usd", "Return_On_Capital", "Roe", "Pe"]

# loop through Box-Muller for each data point for each predictor
for col in feature_cols:
    for i in np.arange(len(df_50)):
        # 1. Use U1 as current data value. Generate U2, which is Unif(0, 1)

        u1s, u2s = (
            df_50.loc[i, col],
            np.random.uniform(low=0.0, high=1.0, size=1)[0],
        )  # X.loc[i, col]

        # 2. Transform U1 to s

        ss = -np.log(u1s)

        # 3. Transform U2 to theta

        thetas = 2 * np.pi * u2s

        # 4. Convert s to r

        rs = np.sqrt(2 * ss)

        # 5. Calculate x and y from r and theta

        xs, ys = rs * np.cos(thetas), rs * np.sin(thetas)

        # 6. Store only one of the Gaussian derived values

        df50_norm.loc[i, col] = ys

Cần xem các hàm hợp lý cho trước nhóm mục tiêu để đánh giá các biến dự báo có thể mạnh hay yếu đến đâu. Ta làm việc này bằng đoạn mã dưới đây:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)

i = 0

sns.kdeplot(ax=axes[0, i], data=df50_norm, x=feature_cols[i], hue="R1M_Usd_C")

i = 1

sns.kdeplot(ax=axes[0, i], data=df50_norm, x=feature_cols[i], hue="R1M_Usd_C")

i = 2

sns.kdeplot(ax=axes[0, i], data=df50_norm, x=feature_cols[i], hue="R1M_Usd_C")

i = 0

sns.kdeplot(ax=axes[1, i], data=df50_norm, x=feature_cols[i + 3], hue="R1M_Usd_C")

i = 1

sns.kdeplot(ax=axes[1, i], data=df50_norm, x=feature_cols[i + 3], hue="R1M_Usd_C")

i = 2

sns.kdeplot(ax=axes[1, i], data=df50_norm, x=feature_cols[i + 3], hue="R1M_Usd_C")

**Hình 2: Hàm hợp lý của cả 6 đặc trưng cho trước nhóm mục tiêu tương ứng**

Nhìn vào kết quả ở hình 2 và so sánh với hình 1, ta thấy sức dự báo của các đặc trưng khá yếu do các hàm hợp lý chồng lấn đáng kể. Do đó ta có thể kỳ vọng hiệu suất của bộ phân loại sẽ kém.

Tiếp theo ta nạp thư viện cần thiết để chia tập dữ liệu thành tập huấn luyện và tập kiểm tra, lần lượt để huấn luyện và đánh giá bộ phân loại. Ta dùng tỷ lệ chia 80/20 cho huấn luyện/kiểm tra. Ta chỉ xét các đặc trưng nêu trên.

In [ ]:
# split dataset in features and target variable

X = df50_norm[feature_cols]  # Transformed Features

y = df50_norm["R1M_Usd_C"]  # Target variable

# Split dataset into training set and test set 80/20

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Ta khớp bộ phân loại Naïve Bayes trên tập huấn luyện và thu các dự báo trên tập kiểm tra chưa từng thấy.

In [ ]:
# initialize and fit classifier to training set
from sklearn.naive_bayes import GaussianNB

classifier = GaussianNB()

classifier.fit(X_train, y_train)

# get the predictions

y_pred = classifier.predict(X_test)

# Extract the probabilities of Target Class = 1.

y_predprobs = classifier.predict_proba(X_test)

probs = y_predprobs[:, 1]

# print the accuracy

from sklearn import metrics

# Model Accuracy, how often is the classifier correct?

print("Accuracy:", metrics.accuracy_score(y_test, y_pred))

Có thể thu được đường cong ROC và AUC bằng đoạn mã dưới đây. Cũng nên vẽ thêm bộ phân loại không có kỹ năng (no-skill), hay đoán ngẫu nhiên, vào biểu đồ để so sánh với bộ phân loại Naïve Bayes.

In [ ]:
# plot the roc curve for the model
from sklearn.metrics import roc_auc_score, roc_curve

# generate a no skill or random guess prediction

ns_probs = [0 for _ in range(len(y_test))]

# calculate roc curves

ns_fpr, ns_tpr, _ = roc_curve(y_test, ns_probs)

# calculate scores

ns_auc = roc_auc_score(y_test, ns_probs)  # random guess

tree_auc = roc_auc_score(y_test, probs)  # tree classifier

# summarize scores

print("No Skill: ROC AUC=%.3f" % (ns_auc))

print("Clasifier: ROC AUC=%.3f" % (tree_auc))

plt.plot(ns_fpr, ns_tpr, linestyle="--", linewidth=4, label="No Skill")

plt.plot(
    ns_fpr,
    ns_tpr,
    linewidth=2,
    label="NB Classifier",
)

# axis labels

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

# show the legend

plt.legend()

# show the plot

plt.show()

**Hình 3: Đường cong ROC của bộ phân loại Naïve Bayes cùng với mô hình đoán ngẫu nhiên (không có kỹ năng)**

Đường cong ROC tương ứng được thể hiện ở hình 3. AUC của mô hình này khoảng ~50%, qua đó xác nhận nghi ngờ của ta về sức dự báo yếu của nó.

## **2. Kết luận**

Ví dụ trong bài giúp ta hiểu bộ phân loại Naïve Bayes qua thực tế áp dụng. Tuy nhiên, hiệu suất của mô hình không tốt như ta mong đợi. Bài này tập trung chỉ ra cách áp dụng Naïve Bayes vào một bài toán phân loại thực tế với các biến dự báo liên tục, và Naïve Bayes có thể dễ dàng áp dụng thế nào cho các biến phân loại. Để có mô hình tốt hơn, cần có các biến dự báo mạnh hơn, tức là các biến giải thích có phân phối hợp lý khác nhau giữa các nhóm mục tiêu. Phân tích dữ liệu khám phá, chọn đặc trưng và kỹ thuật đặc trưng có tầm quan trọng không thể xem nhẹ khi mô hình hóa.

Ở phần tiếp theo, ta sẽ xem các thuật toán dựa trên cây, một kỹ thuật mô hình hóa khác. Mô hình Naïve Bayes giả định các đặc trưng độc lập có điều kiện với biến mục tiêu; còn các thuật toán dựa trên cây không đòi hỏi giả định nào về phân phối nền. Cả Naïve Bayes và cây đều không cần co giãn thang đo đặc trưng, dù các thuật toán dựa trên cây không nhạy với giá trị ngoại lai. Cách dự báo của cây là đặt các ranh giới quyết định trong dữ liệu, trong khi Naïve Bayes nhắm mô hình hóa cách dữ liệu được sinh ra. Chúng lần lượt được gọi là mô hình phân biệt (discriminative) và mô hình sinh (generative). Ta sẽ tìm hiểu cây kỹ hơn ở bài tiếp theo.

**Tài liệu tham khảo**

- Box, G. E. P., and Mervin E. Muller. "A Note on the Generation of Random Normal Deviates." *The Annals of Mathematical Statistics*, vol. 29, no. 2, 1958, pp. 610–11, https://doi.org/10.1214/aoms/1177706645.

- Coqueret, Guillaume, and Tony Guida. *Machine Learning for Factor Investing: R Version.* Financial Mathematics Series. Chapman and Hall/CRC, 2020.